# Demos for analyzing World Color Survey (WCS)

COG 260: Data, Computation, and The Mind (Yang Xu)

Data source: https://linguistics.berkeley.edu/wcs/data.html

______________________________________________

Import helper function file for WCS data analysis.

In [ ]:
from wcs_helper_functions import *

Import relevant Python libraries.

In [ ]:
import numpy as np
from scipy import stats
from random import random
%matplotlib inline

## Demo 1: Import stimulus (color chip) information in [Munsell space](https://en.wikipedia.org/wiki/Munsell_color_system)

> Stimuli were 330 color chips in Munsell space, each defined along lightness and hue dimensions.

> Each color chip has an index _(from 1 to 330)_ and a coordinate (lightness *(alphabet)*, hue *(integer)*).

In the following section, you will learn how to convert from **(a) index to coordinate** and **(b) coordinate to index**.

______________________________________________

Load chip information in Munsell space. 

`munsellInfo` is a 2-element tuple with dictionary elements.

In [ ]:
munsellInfo = readChipData('./WCS_data_core/chip.txt');

### (a) Index &rarr; Coordinate

Access the second dictionary in `munsellInfo`.

In [ ]:
indexCoord = munsellInfo[1]

`indexCoord` is a dictionary with **index _(key)_ &rarr; coordinate _(value)_** pairs. For example, to retrieve the Munsell coordinate _(lightness, hue)_ for chip with numerical index 1:

In [ ]:
print(indexCoord[1])

You can also uncomment the following to display full stimulus information (long).

In [ ]:
# print(indexCoord)

### (b) Coordinate &rarr; Index

Access the first dictionary in `munsellInfo`.

In [ ]:
coordIndex = munsellInfo[0]

`coordIndex` is a dictionary with **coordinate _(key)_ &rarr; index _(value)_** pairs. For example, to access numerical index for the color chip at Munsell coordinate _(D, 11)_:

In [ ]:
print(coordIndex['D11'])

You can also uncomment the following to display full stimulus information (long).

In [ ]:
# print(coordIndex)

## Demo 2: Import stimulus information in [CIELAB space](https://en.wikipedia.org/wiki/Lab_color_space)

> Each of the 330 stimuli can also be mapped to the 3D CIELAB space, where the dimensions are `l` _(lightness)_, `a`, & `b` _(color opponency)_.

> CIELAB Coordinates have a **one-to-one** correspondence with Munsell index, which ranges from 1 to 330.

______________________________________________

Load chip coordinates in CIELAB. 

`cielabCoord` is a dictionary with **index _(key)_ &rarr; CIELAB Coordinate _(value)_** pairs.

In [ ]:
cielabCoord = readClabData('./WCS_data_core/cnum-vhcm-lab-new.txt')

For example, to obtain the CIELAB coordinates for chip with numerical index 1:

In [ ]:
print(cielabCoord[1])

## Demo 3: Import color naming data
    
> Each of the 330 color chips was named by speakers of 110 different languages.

______________________________________________

Load naming data. 

`namingData` is a hierarchical dictionary organized as follows:

**language _(1 - 110)_ &rarr; speaker _(1 - *range varies per language*)_ &rarr; chip index _(1 - 330)_ &rarr; color term**

In [ ]:
namingData = readNamingData('./WCS_data_core/term.txt')

For example, to obtain naming data from language 1 and speaker 1 for all 330 color chips:

In [ ]:
namingData[1][1]; # remove semicolon to see data in full

For example, to see how many speakers language 1 has:

In [ ]:
len(namingData[1])

## Demo 4: Import color foci data
    
> Apart from naming the color chips, each speaker also pointed to foci color chips for each color term they had used.

> **Note**: A single color term may have multiple foci locations.

______________________________________________

Load foci data. 

`fociData` is a hierarchical dictionary organized as follows: 

**language _(1 - 110)_ &rarr; speaker _(1 - *range varies per language*)_ &rarr; color term &rarr; foci coordinates**

In [ ]:
fociData = readFociData('./WCS_data_core/foci-exp.txt');

For example, to obtain foci data for language 1 and speaker 1, where each entry shows foci locations for given term: 

In [ ]:
fociData[1][1]

In the above example, foci for term 'LF' is located at coordinate _(A, 0)_ in the Munsell chart.

## Demo 5: Import speaker demographic information

> Most speakers' age _(integer)_ and gender _(M/F)_ information was recorded.

______________________________________________

Load speaker information.

`speakerInfo` is a hierarchical dictionary organized as follows:

**language &rarr; speaker &rarr; (age, gender)**

In [ ]:
speakerInfo = readSpeakerData('./WCS_data_core/spkr-lsas.txt')

For example, uncomment the following line to access _(age, gender)_ information for all speakers from language 1:

In [ ]:
# speakerInfo[1]

For example, uncomment the following line to access _(age, gender)_ information for speaker 1 from language 1:

In [ ]:
# speakerInfo[1][1]

## Demo 6: Visualize color naming from an individual speaker

> Naming patterns from a speaker can be visualized in the stimulus palette _(Munsell space)_.

______________________________________________

Extract an example speaker datum from an example language.

In [ ]:
lg61_spk5 = namingData[61][5]

Extract color terms used by that speaker.

In [ ]:
terms = lg61_spk5.values()

Encode the color terms into random numbers (for plotting purposes).

In [ ]:
encoded_terms = map_array_to(terms, generate_random_values(terms))

Visualize the color naming pattern for that speaker&mdash;each color patch corresponds to extension of a color term. Color scheme is randomized, but the partition of the color space is invariant.

In [ ]:
plotValues(encoded_terms)

**Note**: `plotValues()` is a generic function for visualizing various kinds of information on the chart, suited to needs.

Now you are in a position to start exploring this data set - enjoy!

In [ ]:
# count the number of established color terms of the languages, which we define a language complexity
# for each language, keep record of how many times a term gloss appears as a term used by a speaker to name a Munsell chip
color_words_across_all_languages = dict()
total_languages = len(namingData)
language = 1
while language < total_languages + 1:
    speakers = namingData[language]
    color_words_per_language = dict()
    # print(len(speakers))
    for speaker in speakers:
        speaker_terms = set(namingData[language][speaker].values())
        for color in speaker_terms:
            if color not in color_words_per_language:
                color_words_per_language[color] = 1
            else:
                color_words_per_language[color] += 1
    # print(language_color_words)
    # number of established terms = the terms that majority of the speakers use
    num_establish_terms = 0
    for color_word in color_words_per_language.keys():
        if color_word != '*' and color_words_per_language[color_word] >= 0.5 * len(speakers):
            num_establish_terms += 1
    color_words_across_all_languages['language ' + str(language)] = num_establish_terms
    language += 1
print(color_words_across_all_languages)

In [ ]:
# now we want to examine color naming consistency across the 330 Munsell chips for
# so we will check what does each speaker in each language name a chip as.

total_chips = 330 # assumes all speakers in every language is asked to name all 330 Munsell chips
color_naming_consistency_across_languages = dict()
# across the 110 languages
for language in range(1, total_languages + 1):
    # across the speakers in the language
    speakers = namingData[language]
    # keep track of color naming of all the 330 Munsell chips by the speakers in the language
    color_naming_all_chips = dict()
    # across the 330 chips
    for chip in range(1, total_chips + 1):
        # store the color naming for each chip
        chip_naming = []
        color_naming_per_chip = dict()
        for speaker in range(1, len(speakers) + 1):
            chip_naming.append(namingData[language][speaker][chip])
        # print(chip_naming)

        # store the proportion of the speakers name each chip with a color word
        for color in chip_naming:
            if color not in color_naming_per_chip:
                color_naming_per_chip[color] = 1/len(speakers)
            else:
                color_naming_per_chip[color] += 1/len(speakers)
        # print(color_naming_per_chip)
        color_naming_all_chips['chip ' + str(chip)] = color_naming_per_chip
    # print(color_naming_all_chips)

    # quantify the consistency in color naming for each language
    language_consistency = sum(max(chip.values()) for chip in color_naming_all_chips.values())/len(color_naming_all_chips)
    # print(language_consistency)
    color_naming_consistency_across_languages['language ' + str(language)] = language_consistency
print(color_naming_consistency_across_languages)

In [ ]:
import scipy

# now we have to examine the correlation between the language complexity, define as the number of established color terms, and the color naming consistency.
x = [] # language complexity
y = [] # color naming consistency

for _, num in color_words_across_all_languages.items():
    x.append(num)

for _, val in color_naming_consistency_across_languages.items():
    y.append(val)
# plotting language complexity against color naming consistency
plt.figure(figsize=(20, 20))
plt.scatter(x, y)
plt.show()

# the correlation coefficient:
corr = scipy.stats.pearsonr(x, y)
print(corr.statistic)
print(color_words_per_language)

## Experiment 2

In [ ]:
def create_coordinates(s):
    key = str(s).replace(":", "").strip()
    index = coordIndex[key]
    return tuple(float(x) for x in cielabCoord[index])

foci_by_language = {} #chip_foci_identified_per_chip_by_language

for lang_key in fociData:
    foci_by_term = {}
    # term: [list of coords mentioned]
    for speaker in fociData[lang_key]:
        foci = fociData[lang_key][speaker]
        for term in foci:
            if len(foci[term]) > 1:
                c = []
                # convert all of its foci to coordinates
                for chip in foci[term]:
                    c.append(create_coordinates(chip))
                # find the center
                arr = np.array(c)
                f = tuple(arr.mean(axis=0))
            else:
                f = create_coordinates(foci[term][0])
            if term in foci_by_term.keys():
                foci_by_term[term].append(f)
            else:
                foci_by_term[term] = [f]

    foci_by_language[lang_key] = foci_by_term

print(foci_by_language[1])
#foci_by_language is a dictionary of [language] --> [colour terms] --> [chip coordinates]

In [ ]:
df = pd.DataFrame(
    [
        {
            "language": language,
            "term": term,
            "chip": chip,

        }
        for language, terms in foci_by_language.items()
        for term, chips in terms.items()
        for chip in chips

    ],
    columns=["language", "term", "chip"],
)

print(df)

In [ ]:
# For each lang
# for each term
# filter out terms with  less than 2 (?) foci identified
    # find center /column mean
    # take euclidean distance from each foci to mean, square and sum, divide by length of foci list
lang_agreement = {}
for lang in foci_by_language:
    term_agreement = []
    for term in foci_by_language[lang]:
        arr = np.array(foci_by_language[lang][term])
        center = arr.mean(axis=0)
        distance = []
        for coord in foci_by_language[lang][term]:
            distance.append(spatial.distance.euclidean(center, coord))
        distance = np.array(distance)
        term_agreement.append(distance.mean())
    lang_agreement[lang] = np.array(term_agreement).mean()

print(lang_agreement)







